# <center style="font-family: consolas; font-size: 32px; font-weight: bold;">  Hands-On LangChain for LLM Applications Development: Prompt Templates </center>

# <center style="font-family: consolas; font-size: 25px; font-weight: bold;">  Understanding LangChain Prompt Templates (OpenRouter + Google Colab Edition) </center>
***

By prompting an LLM or large language model, it is possible to develop complex AI applications much faster than ever before. However, an application can require prompting an LLM multiple times and parsing its output, so a lot of glue code must be written.

LangChain makes this development process much easier by using an easy set of abstractions to do this type of operation and by providing prompt templates. In this notebook, we will cover prompt templates, why it is important, and how to use them effectively, explained with practical examples.

**Note:** This version has been adapted to run on **Google Colab** and use **OpenRouter** instead of calling OpenAI directly or using Kaggle Secrets, so you can use any model available on OpenRouter (GPT, Claude, Llama, Gemini, etc.) with a single API key.

#### <a id="top"></a>
# <div style="box-shadow: rgb(60, 121, 245) 0px 0px 0px 3px inset, rgb(255, 255, 255) 10px -10px 0px -3px, rgb(31, 193, 27) 10px -10px, rgb(255, 255, 255) 20px -20px 0px -3px, rgb(255, 217, 19) 20px -20px, rgb(255, 255, 255) 30px -30px 0px -3px, rgb(255, 156, 85) 30px -30px, rgb(255, 255, 255) 40px -40px 0px -3px, rgb(255, 85, 85) 40px -40px; padding:20px; margin-right: 40px; font-size:30px; font-family: consolas; text-align:center; display:fill; border-radius:15px; color:rgb(60, 121, 245);"><b>Table of contents</b></div>

<div style="background-color: rgba(60, 121, 245, 0.03); padding:30px; font-size:15px; font-family: consolas;">
<ul>
    <li><a href="#1" target="_self" rel=" noreferrer nofollow">1. Setting Up Working Environment with OpenRouter on Colab </a> </li>
    <li><a href="#2" target="_self" rel=" noreferrer nofollow">2. Prompt Template using LangChain </a></li>
    <li><a href="#3" target="_self" rel=" noreferrer nofollow">3. Why do We Need LangChain Prompt Templates? </a></li>
</ul>
</div>

***


# <div style="box-shadow: rgba(0, 0, 0, 0.16) 0px 1px 4px inset, rgb(51, 51, 51) 0px 0px 0px 3px inset; padding:20px; font-size:32px; font-family: consolas; text-align:center; display:fill; border-radius:15px;  color:rgb(34, 34, 34);"> <b> 1. Setting Up Working Environment with OpenRouter on Colab </b></div>

We're going to use [OpenRouter](https://openrouter.ai) instead of connecting directly to OpenAI. OpenRouter provides an endpoint fully compatible with the OpenAI SDK (`https://openrouter.ai/api/v1`), and gives you access to many models (OpenAI, Anthropic, Google, Meta...) with a single API key.

Let's install the required libraries first.


In [11]:
# Install required libraries (Colab)
!pip install -q openai langchain langchain-community langchain-core


### OpenRouter API Key

Create an account at [openrouter.ai](https://openrouter.ai/keys) and get an API key.

On Colab, I recommend storing the key in **Secrets** (the 🔑 icon on the left sidebar) under the name `OPENROUTER_API_KEY` — this is the safest option. If the secret isn't found, you'll be prompted to enter it manually (and the value won't be shown as you type).


In [12]:
import os

try:
    # If you're running this on Google Colab
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
except Exception:
    OPENROUTER_API_KEY = None

if not OPENROUTER_API_KEY:
    import getpass
    OPENROUTER_API_KEY = getpass.getpass("Enter your OpenRouter API key: ")

os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY


In [13]:
from openai import OpenAI

# OpenAI SDK client, but pointed at the OpenRouter endpoint
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)


Next, we'll define the model we're going to use. You can pick any model available on OpenRouter (see the full list here: https://openrouter.ai/models).

Here we'll use `openai/gpt-3.5-turbo` as an example (a direct replacement for the original `gpt-3.5-turbo`), but you can change it to any other model such as `anthropic/claude-3.5-sonnet`, `google/gemini-2.0-flash-001`, `meta-llama/llama-3.1-8b-instruct`, and more.


In [14]:
llm_model = "openai/gpt-oss-20b:free"


Next, we will define a helper function that will take the input prompt and return the response.


In [15]:
def get_completion(prompt,
                    llm_model=llm_model,
                    temperature=0,
                    max_tokens=500):

    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=llm_model,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content


Let's start with an example where you get an email from a customer in a language other than formal English. To make sure the example is understandable, the other language we will use is the English pirate language:


In [16]:
customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse,\
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""


We will ask the LLM to translate the text to formal English in a calm and respectful tone. I will set the style to American English in a calm and respectful tone. I will specify the prompt using an f-string with the instructions, translate the text that is delimited by triple backticks into style, and then plug in these two styles. This generates a prompt that says translate the text.


In [17]:
style = """American English \
in a calm and respectful tone
"""

prompt = f"""Translate the text \
that is delimited by triple backticks
into a style that is {style}.
text: ```{customer_email}```
"""

print(prompt)


Translate the text that is delimited by triple backticks 
into a style that is American English in a calm and respectful tone
.
text: ```
Arrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie! And to make matters worse,the warranty don't cover the cost of cleaning up me kitchen. I need yer help right now, matey!
```



Let's see what the response is:


In [18]:
response = get_completion(prompt)
response


'I’m upset that my blender lid flew off and splattered my kitchen walls with smoothie. To make matters worse, the warranty doesn’t cover the cost of cleaning up my kitchen. I need your help right now.'

That sounds very nice and calm. Therefore if you have different customers writing reviews in different languages, not just English pirates, but French, German, Japanese, and so on, you can imagine having to generate a whole sequence of prompts to generate such translations. Let's look at how we can do this in a more convenient, way using **LangChain**.


<a id="2"></a>
# <div style="box-shadow: rgba(0, 0, 0, 0.16) 0px 1px 4px inset, rgb(51, 51, 51) 0px 0px 0px 3px inset; padding:20px; font-size:32px; font-family: consolas; text-align:center; display:fill; border-radius:15px;  color:rgb(34, 34, 34);"> <b> 2. Prompt Template using LangChain </b></div>


Let's start with importing chat OpenAI. This is LangChain's abstraction for a Chat Completions API endpoint — and since OpenRouter is OpenAI-compatible, we just point `openai_api_base` at OpenRouter instead. I will set the temperature parameter to be equal to zero to make the output a little bit less random.


In [21]:
!pip install -q langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.1/122.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 17.0 MB/s eta 0:00:00


In [22]:
from langchain_openai import ChatOpenAI

# To control the randomness and creativity of the generated
# text by an LLM, use temperature = 0.0

chat = ChatOpenAI(
    temperature=0.0,
    model=llm_model,
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base="https://openrouter.ai/api/v1",
)

We will define the template string as follows. Translate the text delimited by triple backticks into a style that is style, and then here's the text.


In [23]:
template_string = """Translate the text \
that is delimited by triple backticks \
into a style that is {style}. \
text: ```{text}```
"""


To repeatedly reuse this template, we have to import LangChain's chat prompt template, and then, let me create a prompt template using that template string that we just wrote above.


In [24]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template(template_string)


From the prompt template, you can extract the original prompt, and it realizes that this prompt has two input variables, the style, and the text, shown here with the curly braces.


In [25]:
prompt_template.messages[0].prompt


PromptTemplate(input_variables=['style', 'text'], input_types={}, partial_variables={}, template='Translate the text that is delimited by triple backticks into a style that is {style}. text: ```{text}```\n')

We can also print the input variables out, and you can see that it realizes it has two input variables the style and text


In [26]:
prompt_template.messages[0].prompt.input_variables


['style', 'text']

Now, let's specify the style. This is a style that I want the customer message to be translated to, so I'm going to call this customer style, and here's my same customer email as before.


In [27]:
customer_style = """American English \
in a calm and respectful tone
"""

customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse, \
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""


If I create customer messages, this will generate the prompt and will pass this large language model to get a response.

If you want to look at the types, the customer message is a list, and if you look at the first element of the list, this is more or less the prompt that you would expect this to be creating.


In [28]:
customer_messages = prompt_template.format_messages(
                    style=customer_style,
                    text=customer_email)

print(type(customer_messages))
print(type(customer_messages[0]))


<class 'list'>
<class 'langchain_core.messages.human.HumanMessage'>


Lastly, let's pass this prompt to the LLM, so I'm going to call chat, which we had set earlier, as a reference to the OpenRouter endpoint (which is compatible with ChatGPT and many other models), and, if we print out the customer responses content, then, it gives you back this text translated from English pirate to polite American English


In [29]:
# Call the LLM to translate to the style of the customer message
customer_response = chat.invoke(customer_messages)

print(customer_response.content)


I’m upset that my blender lid flew off and splattered my kitchen walls with smoothie. To make matters worse, the warranty doesn’t cover the cost of cleaning up my kitchen. I need your help right now.


Of course, you can imagine other use cases where the customer emails are in other languages and this too can be used to translate the messages for an English-speaking to understand and reply to.

So let's say, an English-speaking customer service agent writes this and says,


In [30]:
service_reply = """Hey there customer, \
the warranty does not cover \
cleaning expenses for your kitchen \
because it's your fault that \
you misused your blender \
by forgetting to put the lid on before \
starting the blender. \
Tough luck! See ya!
"""


But let's say this is what a customer service agent wants. We are going to specify that the service message is going to be translated to this pirate style. So we want it to be in a polite tone that speaks in English pirate. And because we previously created that prompt template, the cool thing is, that we can now reuse that prompt template and specify that the output style we want is this service style pirate and the text is this service reply.



In [31]:
service_style_pirate = """\
a polite tone \
that speaks in English Pirate\
"""


And if we do that, that's the prompt.


In [32]:
service_messages = prompt_template.format_messages(
    style=service_style_pirate,
    text=service_reply)

print(service_messages[0].content)


Translate the text that is delimited by triple backticks into a style that is a polite tone that speaks in English Pirate. text: ```Hey there customer, the warranty does not cover cleaning expenses for your kitchen because it's your fault that you misused your blender by forgetting to put the lid on before starting the blender. Tough luck! See ya!
```



And if we prompt, ChatGPT, this is the response it gives us back


In [33]:
service_response = chat.invoke(service_messages)
print(service_response.content)


Ahoy, esteemed customer.  
We regret to inform ye that the warranty does not cover the cleaning expenses for yer kitchen, as the mishap arose from yer own misuse of the blender—specifically, forgetting to secure the lid before setting it in motion. Unfortunately, we cannot cover this cost.  

We wish ye fair winds and calm seas. Farewell!


<a id="3"></a>
# <div style="box-shadow: rgba(0, 0, 0, 0.16) 0px 1px 4px inset, rgb(51, 51, 51) 0px 0px 0px 3px inset; padding:20px; font-size:32px; font-family: consolas; text-align:center; display:fill; border-radius:15px;  color:rgb(34, 34, 34);"> <b> 3. Why do We Need LangChain Prompt Templates? </b></div>


We are using prompt templates instead of just an f-string prompt because as you build sophisticated applications, prompts can be quite long and detailed. Prompt templates are a useful abstraction to help you reuse good prompts when you can.

This is an example of a relatively long prompt to decide if a student's solution is correct or not. And a prompt like this can be quite long, in which you can ask the LLM to first solve the problem, and then have the output in a certain format, and the output in a certain format.


In [34]:
prompt = f"""
Determine if the student's solution is correct or not.

Question:
I'm building a solar power installation and I need \
 help working out the financials.
- Land costs $100 / square foot
- I can buy solar panels for $250 / square foot
- I negotiated a contract for maintenance that will cost \
me a flat $100k per year, and an additional $10 / square \
foot
What is the total cost for the first year of operations
as a function of the number of square feet.

Student's Solution:
Let x be the size of the installation in square feet.
Costs:
1. Land cost: 100x
2. Solar panel cost: 250x
3. Maintenance cost: 100,000 + 100x
Total cost: 100x + 250x + 100,000 + 100x = 450x + 100,000
"""


<>:30: SyntaxWarning: invalid escape sequence '\ '
<>:30: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_741/2803865163.py:30: SyntaxWarning: invalid escape sequence '\ '


LangChain prompt makes it easier to reuse a prompt like this. Also, as you will see in the next articles LangChain provides prompts for some common operations, such as summarization, question answering, connecting to SQL databases, or connecting to different APIs. So by using some of LangChain's built-in prompts, you can quickly get an application working without needing to, engineer your prompts.


### Summary of changes made in this version

- Replaced `kaggle_secrets` and direct OpenAI calls with **OpenRouter** (`base_url = https://openrouter.ai/api/v1`).
- Used `google.colab.userdata` to securely read the API key on Colab (with a `getpass` fallback if not set).
- Updated `ChatOpenAI` to point at `openrouter.ai` instead of `api.openai.com`.
- Replaced the old (deprecated) `chat(messages)` call with `chat.invoke(messages)` to match the latest LangChain versions.
- You can easily switch to any other model available on OpenRouter (GPT, Claude, Gemini, Llama, etc.).

# <div style="box-shadow: rgba(240, 46, 170, 0.4) -5px 5px inset, rgba(240, 46, 170, 0.3) -10px 10px inset, rgba(240, 46, 170, 0.2) -15px 15px inset, rgba(240, 46, 170, 0.1) -20px 20px inset, rgba(240, 46, 170, 0.05) -25px 25px inset; padding:20px; font-size:30px; font-family: consolas; display:fill; border-radius:15px; color: rgba(240, 46, 170, 0.7)"> <b> ༼⁠ ⁠つ⁠ ⁠◕⁠‿⁠◕⁠ ⁠༽⁠つ Thank You!</b></div>
